# Emotion Detection in Text-based Conversations

In [ ]:
!pip install transformers streamlit torch sklearn pandas matplotlib

import torch
from transformers import BertTokenizer, BertForSequenceClassification
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
# ---- Cell 2: Data Loading ----
df = pd.read_csv('emotion_dataset.csv')  # Ensure columns: text, emotion
df.head()

In [ ]:
# ---- Cell 3: Data Preprocessing ----
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['emotion_label'] = le.fit_transform(df['emotion'])

X = df['text']
y = df['emotion_label']

In [ ]:
# ---- Cell 4: Train/Test Split ----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_labels = len(le.classes_)

In [ ]:
# ---- Cell 5: Tokenization ----
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128, return_tensors='pt')

In [ ]:
# ---- Cell 6: Model Setup ----
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

In [ ]:
# ---- Cell 7: Training Loop (Simplified) ----
from torch.optim import AdamW

train_encodings = preprocess(X_train)
test_encodings = preprocess(X_test)

train_labels = torch.tensor(y_train.values)
test_labels = torch.tensor(y_test.values)

optim = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(1):
    optim.zero_grad()
    outputs = model(**train_encodings, labels=train_labels)
    loss = outputs.loss
    loss.backward()
    optim.step()
    print(f"Epoch {epoch}: Loss = {loss.item()}")

In [ ]:
# ---- Cell 8: Evaluation ----
model.eval()
with torch.no_grad():
    outputs = model(**test_encodings)
    preds = torch.argmax(outputs.logits, axis=1).numpy()

print(classification_report(y_test, preds, target_names=le.classes_))
conf_mat = confusion_matrix(y_test, preds)

plt.figure(figsize=(8,6))
plt.imshow(conf_mat, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ---- Cell 9: Save Model ----
model.save_pretrained('./emotion_detection_model')
tokenizer.save_pretrained('./emotion_detection_model')

In [ ]:
# ---- Cell 10: Streamlit Realtime Prediction App ----
# Save this as app.py, then run: streamlit run app.py

import streamlit as st
from transformers import BertTokenizer, BertForSequenceClassification
import torch

st.title("Realtime Emotion Detection in Text")
st.subheader("Enter your message below:")

user_input = st.text_area("Text Input", "")
if st.button("Detect Emotion"):
    tokenizer = BertTokenizer.from_pretrained('./emotion_detection_model')
    model = BertForSequenceClassification.from_pretrained('./emotion_detection_model')
    model.eval()
    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, max_length=128)
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    emotion_list = list(le.classes_)
    st.write(f"**Predicted Emotion:** {emotion_list[pred]}")